# Huấn luyện LoRA trên Google Colab & Đăng ký vào Hệ thống

Đây là notebook để huấn luyện một adapter LoRA mới từ file văn bản, tải nó lên Hugging Face Hub, và đăng ký nó với hệ thống TWP-Omega-VInfinity của bạn.

### Bước 1: Cài đặt các thư viện

In [ ]:
!pip install -q transformers==4.41.2 datasets==2.19.0 peft==0.10.0 bitsandbytes==0.43.1 trl==0.8.6 qdrant-client==1.8.2 sentence-transformers==2.7.0 huggingface_hub==0.23.0

### Bước 2: Cấu hình và Đăng nhập

1.  **Thêm Secrets:** Ở thanh bên trái, bấm vào biểu tượng chìa khóa (Secrets) và thêm các secret sau:
    *   `HF_TOKEN_WRITE`: Token Hugging Face của bạn với quyền **write**.
    *   `QDRANT_URL`: URL cluster Qdrant Cloud của bạn.
    *   `QDRANT_API_KEY`: API key của Qdrant Cloud.

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

# Đăng nhập vào Hugging Face
HF_TOKEN = userdata.get('HF_TOKEN_WRITE')
login(token=HF_TOKEN)

# --- Cấu hình --- #
QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')

# !!! THAY THẾ BẰNG USERNAME CỦA BẠN !!!
HF_USERNAME = "your-huggingface-username"
BASE_MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

### Bước 3: Tải lên và Chuẩn bị Dữ liệu Huấn luyện

In [ ]:
from google.colab import files
from datasets import Dataset
import uuid

print("Vui lòng tải lên file văn bản (.txt) chứa kiến thức bạn muốn huấn luyện.")
uploaded = files.upload()

if not uploaded:
  raise ValueError("Không có file nào được tải lên!")

file_path = list(uploaded.keys())[0]
USER_ID = input("Nhập User ID của bạn: ")
SOURCE_ID = input("Nhập Source ID cho dữ liệu này: ")

with open(file_path, 'r') as f:
    content = f.read()

# Chuyển đổi văn bản thô thành định dạng instruction
def create_instruction_dataset(text):
    # Một phương pháp đơn giản: chia thành các đoạn và tạo câu hỏi-đáp
    chunks = [p.strip() for p in text.split('\n\n') if p.strip()]
    instructions = []
    for chunk in chunks:
        instructions.append({
            "text": f"### Instruction:\nSummarize the following text.\n\n### Input:\n{chunk}\n\n### Response:"
        })
    return Dataset.from_list(instructions)

dataset = create_instruction_dataset(content)

### Bước 4: Huấn luyện LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=transformers.TrainingArguments(per_device_train_batch_size=1, gradient_accumulation_steps=4, warmup_steps=2, max_steps=10, learning_rate=2e-4, fp16=True, logging_steps=1, output_dir="outputs")
)

trainer.train()
print('--- HUẤN LUYỆN HOÀN TẤT ---')

### Bước 5: Tải LoRA lên Hugging Face Hub

In [ ]:
LORA_REPO_NAME = f"lora-{USER_ID}-{SOURCE_ID}-{str(uuid.uuid4())[:8]}"
trainer.model.push_to_hub(LORA_REPO_NAME)
print(f"Đã tải LoRA lên Hub tại: {HF_USERNAME}/{LORA_REPO_NAME}")

### Bước 6: Đăng ký LoRA vào Qdrant Cloud

In [ ]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

print("Đang tạo vector đại diện cho LoRA...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
content_vector = embedding_model.encode(content).tolist()

qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

qdrant_client.upsert(
    collection_name="lora_registry",
    points=[
        models.PointStruct(
            id=str(uuid.uuid4()),
            vector=content_vector,
            payload={
                "path": f"{HF_USERNAME}/{LORA_REPO_NAME}", # Quan trọng: đây là repo_id trên Hub
                "user_id": USER_ID,
                "source_id": SOURCE_ID
            }
        )
    ]
)

print('--- HOÀN TẤT! LoRA đã được huấn luyện, tải lên và đăng ký vào hệ thống. ---')